In [62]:
# Cell 1: Modbus connection
from pymodbus.client import ModbusTcpClient
import time as tt
import threading

IP_ADDRESS = '210.119.14.58'
PORT = 502
UNIT_ID = 1

client = ModbusTcpClient(IP_ADDRESS, port=PORT)
connected = client.connect()
print('connected =', connected)


connected = True


Queue full, skipped: green
Queue full, skipped: blue
Queue full, skipped: green


In [63]:
# Cell 2: I/O mapping from Factory I/O driver
IN_MC0_OPENED = 0
IN_MC0_BUSY = 1
IN_MC0_ERROR = 2
IN_PUSHER_FRONT = 3
IN_PUSHER_BACK = 4
IN_MC1_OPENED = 5
IN_MC1_BUSY = 6
IN_MC1_ERROR = 7
IN_SENSOR_0 = 8
IN_SENSOR_1 = 9
IN_SENSOR_2 = 10
IN_SENSOR_3 = 11
IN_SENSOR_4 = 12

REG_MC0_PROGRESS = 0
REG_MC1_PROGRESS = 1

COIL_EMITTER_0 = 0
COIL_MC0_STOP = 1
COIL_MC0_RESET = 2
COIL_MC0_START = 3
COIL_MC0_PRODUCE = 4
COIL_PUSHER_0 = 5
COIL_REMOVER_0 = 6
COIL_EMITTER_1 = 9
COIL_MC1_STOP = 10
COIL_MC1_RESET = 11
COIL_MC1_START = 12
COIL_MC1_PRODUCE = 13
COIL_BELT_2M_0 = 14
COIL_BELT_2M_2 = 15
COIL_BELT_4M_0 = 16
COIL_BELT_4M_1 = 17
COIL_BELT_4M_2 = 18
COIL_BELT_6M_0 = 19
COIL_BELT_6M_1 = 20
COIL_BELT_6M_2 = 21
COIL_BELT_6M_3 = 22
COIL_CURVE_0_CW = 23
COIL_CURVE_3_CW = 24
COIL_CURVE_1_CCW = 25
COIL_CURVE_2_CCW = 26
COIL_CURVE_4_CCW = 27
COIL_BELT_4M_3 = 28


Queue full, skipped: blue
mc0 blue start= 3 produce= 4
Queue full, skipped: green
mc0 busy=True progress= 0


In [64]:
# Cell 3: process settings
CONVEYOR_COILS = [
    COIL_BELT_2M_0, COIL_BELT_2M_2,
    COIL_BELT_4M_0, COIL_BELT_4M_1, COIL_BELT_4M_2, COIL_BELT_4M_3,
    COIL_BELT_6M_0, COIL_BELT_6M_1, COIL_BELT_6M_2, COIL_BELT_6M_3,
    COIL_CURVE_0_CW, COIL_CURVE_3_CW,
    COIL_CURVE_1_CCW, COIL_CURVE_2_CCW, COIL_CURVE_4_CCW,
]

SORT_SENSOR = IN_SENSOR_3
EXIT_SENSOR = IN_SENSOR_4
SORT_SENSOR_CANDIDATES = [IN_SENSOR_2, IN_SENSOR_3, IN_SENSOR_4]
USE_SORT_SENSOR_CANDIDATES = True

EMIT_TIME = 0.35
START_HOLD = 0.5
PRODUCE_HOLD = 6.0
WAIT_BUSY_TIMEOUT = 10.0
FEED_TO_MC0_TIME = 4.5
FEED_TO_MC1_TIME = 5.5
MACHINE_TIMEOUT = 12.0
WAIT_SORT_DONE_TIMEOUT = 35.0
REPEAT_DELAY = 0.5

PUSHER_ACTIVATION_DELAY = 0.0
PUSH_TIME = 0.8
PUSH_EXTEND_TIMEOUT = 2.0
PUSH_RETRACT_TIMEOUT = 2.0
REMOVE_TIME = 0.45
SCAN_TIME = 0.01
MAX_QUEUE_SIZE = 4

process_run = False
machine_repeat_run = False
production_thread = None
sorting_thread = None
machine_repeat_thread = None
product_queue = []
queue_lock = threading.Lock()
sort_done_event = threading.Event()


In [65]:
# Cell 4: Modbus helper functions
def _is_error(result):
    return hasattr(result, 'isError') and result.isError()

def read_input(addr):
    try:
        result = client.read_discrete_inputs(addr, count=1, slave=UNIT_ID)
    except TypeError:
        result = client.read_discrete_inputs(addr, count=1)
    if _is_error(result):
        raise RuntimeError(f'Input {addr} read failed')
    return bool(result.bits[0])

def read_register(addr):
    try:
        result = client.read_input_registers(addr, count=1, slave=UNIT_ID)
    except TypeError:
        result = client.read_input_registers(addr, count=1)
    if _is_error(result):
        raise RuntimeError(f'Input register {addr} read failed')
    return result.registers[0]

def write_coil(addr, value):
    try:
        result = client.write_coil(addr, bool(value), slave=UNIT_ID)
    except TypeError:
        result = client.write_coil(addr, bool(value))
    if _is_error(result):
        raise RuntimeError(f'Coil {addr} write failed')
    return result

def write_coils(addr, values):
    try:
        result = client.write_coils(addr, values, slave=UNIT_ID)
    except TypeError:
        result = client.write_coils(addr, values)
    if _is_error(result):
        raise RuntimeError(f'Coils from {addr} write failed')
    return result

def pulse_coil(addr, seconds=0.5):
    write_coil(addr, True)
    tt.sleep(seconds)
    write_coil(addr, False)


In [66]:
# Cell 5: stop and conveyor functions
def conveyors_on():
    for coil in CONVEYOR_COILS:
        write_coil(coil, True)

def conveyors_off():
    for coil in CONVEYOR_COILS:
        write_coil(coil, False)

def stop_all():
    global process_run, machine_repeat_run
    process_run = False
    machine_repeat_run = False
    write_coils(0, [False] * 30)
    print('All outputs stopped')


In [ ]:
# Cell 6: machining center and actuator functions
MACHINES = {
    'mc0': {'color': 'blue', 'stop': COIL_MC0_STOP, 'reset': COIL_MC0_RESET, 'start': COIL_MC0_START, 'produce': COIL_MC0_PRODUCE, 'busy': IN_MC0_BUSY, 'error': IN_MC0_ERROR, 'progress': REG_MC0_PROGRESS, 'feed_time': FEED_TO_MC0_TIME},
    'mc1': {'color': 'green', 'stop': COIL_MC1_STOP, 'reset': COIL_MC1_RESET, 'start': COIL_MC1_START, 'produce': COIL_MC1_PRODUCE, 'busy': IN_MC1_BUSY, 'error': IN_MC1_ERROR, 'progress': REG_MC1_PROGRESS, 'feed_time': FEED_TO_MC1_TIME},
}

def reset_machine(name):
    m = MACHINES[name]
    write_coil(m['stop'], False)
    write_coil(m['start'], False)
    write_coil(m['produce'], False)
    pulse_coil(m['reset'], 0.7)
    tt.sleep(0.3)

def reset_machines():
    reset_machine('mc0')
    reset_machine('mc1')

def enable_machine(name):
    m = MACHINES[name]
    write_coil(m['stop'], False)
    write_coil(m['start'], True)
    tt.sleep(START_HOLD)

def disable_machine(name):
    m = MACHINES[name]
    write_coil(m['produce'], False)
    write_coil(m['start'], False)

def wait_until_busy(name, timeout=WAIT_BUSY_TIMEOUT):
    m = MACHINES[name]
    end_time = tt.time() + timeout
    while tt.time() < end_time:
        busy = read_input(m['busy'])
        progress = read_register(m['progress'])
        if busy:
            print(name, 'busy=True progress=', progress)
            return True
        tt.sleep(0.2)
    print(name, 'busy did not start, progress=', read_register(m['progress']))
    return False

def wait_machine_activity(name, timeout=MACHINE_TIMEOUT):
    m = MACHINES[name]
    start_progress = read_register(m['progress'])
    end_time = tt.time() + timeout
    saw_busy = False
    while tt.time() < end_time:
        busy = read_input(m['busy'])
        progress = read_register(m['progress'])
        if busy:
            saw_busy = True
        if progress != start_progress:
            return True
        if saw_busy and not busy:
            return True
        tt.sleep(0.1)
    return False

def run_machine(name):
    m = MACHINES[name]
    print(name, m['color'], 'start=', m['start'], 'produce=', m['produce'])
    if read_input(m['error']):
        reset_machine(name)
    enable_machine(name)
    write_coil(m['produce'], True)
    busy_started = wait_until_busy(name)
    if not busy_started:
        tt.sleep(PRODUCE_HOLD)
    moved = wait_machine_activity(name)
    write_coil(m['produce'], False)
    print(name, 'done busy_started=', busy_started, 'activity=', moved, 'progress=', read_register(m['progress']))
    return busy_started or moved

def emit_mc0_blue():
    pulse_coil(COIL_EMITTER_0, EMIT_TIME)

def emit_mc1_green():
    pulse_coil(COIL_EMITTER_1, EMIT_TIME)

def feed_machine_and_run(name, emit_func):
    conveyors_on()
    reset_machine(name)
    enable_machine(name)
    emit_func()
    tt.sleep(MACHINES[name]['feed_time'])
    run_machine(name)
    return MACHINES[name]['color']

def feed_mc0_and_run():
    return feed_machine_and_run('mc0', emit_mc0_blue)

def feed_mc1_and_run():
    return feed_machine_and_run('mc1', emit_mc1_green)

def wait_for_input(addr, expected=True, timeout=2.0):
    end_time = tt.time() + timeout
    while tt.time() < end_time:
        if read_input(addr) == expected:
            return True
        tt.sleep(0.02)
    return False

def finish_pusher_cycle():
    reached_front = wait_for_input(IN_PUSHER_FRONT, True, PUSH_EXTEND_TIMEOUT)
    if not reached_front:
        tt.sleep(PUSH_TIME)
    write_coil(COIL_PUSHER_0, False)
    wait_for_input(IN_PUSHER_BACK, True, PUSH_RETRACT_TIMEOUT)
    print('Pusher cycle done')

def push_blue_product():
    if PUSHER_ACTIVATION_DELAY > 0:
        tt.sleep(PUSHER_ACTIVATION_DELAY)
    print('Pusher ON immediately: blue material')
    write_coil(COIL_PUSHER_0, True)
    finish_pusher_cycle()

def remove_unknown_product():
    print('Remover ON: unknown material only')
    pulse_coil(COIL_REMOVER_0, REMOVE_TIME)

def handle_sorting(color):
    if color == 'blue':
        push_blue_product()
    elif color == 'green':
        print('Green passed: pusher stays OFF')
        write_coil(COIL_PUSHER_0, False) 
    else:
        write_coil(COIL_PUSHER_0, False)
        remove_unknown_product()


In [68]:
# Cell 7: diagnostics and tests
INPUT_NAMES = {
    IN_MC0_OPENED: 'MC0 Opened', IN_MC0_BUSY: 'MC0 Busy', IN_MC0_ERROR: 'MC0 Error',
    IN_PUSHER_FRONT: 'Pusher Front Limit', IN_PUSHER_BACK: 'Pusher Back Limit',
    IN_MC1_OPENED: 'MC1 Opened', IN_MC1_BUSY: 'MC1 Busy', IN_MC1_ERROR: 'MC1 Error',
    IN_SENSOR_0: 'Retro Sensor 0', IN_SENSOR_1: 'Retro Sensor 1', IN_SENSOR_2: 'Retro Sensor 2',
    IN_SENSOR_3: 'Retro Sensor 3 / SORT_SENSOR default', IN_SENSOR_4: 'Retro Sensor 4 / EXIT_SENSOR default',
}

def print_inputs(seconds=5):
    end_time = tt.time() + seconds
    while tt.time() < end_time:
        print([read_input(i) for i in range(13)])
        tt.sleep(0.5)

def read_sort_sensor():
    sensors = SORT_SENSOR_CANDIDATES if USE_SORT_SENSOR_CANDIDATES else [SORT_SENSOR]
    active = [addr for addr in sensors if read_input(addr)]
    return bool(active), active

def monitor_sort_area(seconds=20):
    print('Watch which Retro Sensor turns True when blue reaches the pusher.')
    print('SORT_SENSOR =', SORT_SENSOR, 'candidates =', SORT_SENSOR_CANDIDATES)
    prev = {}
    end_time = tt.time() + seconds
    while tt.time() < end_time:
        current = {
            IN_SENSOR_0: read_input(IN_SENSOR_0), IN_SENSOR_1: read_input(IN_SENSOR_1),
            IN_SENSOR_2: read_input(IN_SENSOR_2), IN_SENSOR_3: read_input(IN_SENSOR_3), IN_SENSOR_4: read_input(IN_SENSOR_4),
            IN_PUSHER_FRONT: read_input(IN_PUSHER_FRONT), IN_PUSHER_BACK: read_input(IN_PUSHER_BACK),
        }
        changed = [f'{INPUT_NAMES[k]}={v}' for k, v in current.items() if prev.get(k) != v]
        if changed:
            print(' | '.join(changed), 'queue=', product_queue)
        prev = current
        tt.sleep(0.05)

def print_machine_status(seconds=5):
    end_time = tt.time() + seconds
    while tt.time() < end_time:
        print(
            f'MC0 blue opened={read_input(IN_MC0_OPENED)}, busy={read_input(IN_MC0_BUSY)}, error={read_input(IN_MC0_ERROR)}, progress={read_register(REG_MC0_PROGRESS)} | '
            f'MC1 green opened={read_input(IN_MC1_OPENED)}, busy={read_input(IN_MC1_BUSY)}, error={read_input(IN_MC1_ERROR)}, progress={read_register(REG_MC1_PROGRESS)}'
        )
        tt.sleep(0.5)

def test_output(coil, seconds=1):
    print(f'Coil {coil} ON')
    write_coil(coil, True)
    tt.sleep(seconds)
    write_coil(coil, False)
    print(f'Coil {coil} OFF')

def test_pusher_only(seconds=1.0):
    print('Testing Pusher 0 coil =', COIL_PUSHER_0)
    write_coil(COIL_PUSHER_0, True)
    tt.sleep(seconds)
    write_coil(COIL_PUSHER_0, False)
    tt.sleep(0.5)
    print('front=', read_input(IN_PUSHER_FRONT), 'back=', read_input(IN_PUSHER_BACK))

def test_machine(name):
    reset_machine(name)
    enable_machine(name)
    run_machine(name)
    print_machine_status(seconds=3)

def force_mc0_test():
    print('MC0 blue force test')
    reset_machine('mc0')
    enable_machine('mc0')
    conveyors_on()
    emit_mc0_blue()
    tt.sleep(FEED_TO_MC0_TIME)
    write_coil(COIL_MC0_PRODUCE, True)
    busy_started = wait_until_busy('mc0')
    if not busy_started:
        tt.sleep(PRODUCE_HOLD)
    print_machine_status(seconds=5)
    write_coil(COIL_MC0_PRODUCE, False)

def force_mc1_test():
    print('MC1 green force test')
    reset_machine('mc1')
    enable_machine('mc1')
    conveyors_on()
    emit_mc1_green()
    tt.sleep(FEED_TO_MC1_TIME)
    write_coil(COIL_MC1_PRODUCE, True)
    busy_started = wait_until_busy('mc1')
    if not busy_started:
        tt.sleep(PRODUCE_HOLD)
    print_machine_status(seconds=5)
    write_coil(COIL_MC1_PRODUCE, False)


In [69]:
# Cell 8: repeated machining-center work
def machine_repeat(mc='both'):
    global machine_repeat_run
    machine_repeat_run = True
    reset_machines()
    conveyors_on()
    print('Repeated machining-center work started')

    try:
        while machine_repeat_run:
            if mc in ('mc0', 'both'):
                feed_mc0_and_run()
                tt.sleep(REPEAT_DELAY)
            if mc in ('mc1', 'both'):
                feed_mc1_and_run()
                tt.sleep(REPEAT_DELAY)
    finally:
        for m in MACHINES.values():
            write_coil(m['start'], False)
            write_coil(m['produce'], False)
        machine_repeat_run = False
        print('Repeated machining-center work stopped')


In [70]:
# Cell 9: queue helpers
def queue_product(color):
    with queue_lock:
        if len(product_queue) < MAX_QUEUE_SIZE:
            product_queue.append(color)
            print('Queued:', color, 'queue=', product_queue)
            return True
        print('Queue full, skipped:', color)
        return False

def pop_product():
    with queue_lock:
        if product_queue:
            return product_queue.pop(0)
    return 'unknown'

def peek_product():
    with queue_lock:
        if product_queue:
            return product_queue[0]
    return 'unknown'


In [71]:
# Cell 10: full process - queue first, then process, blue pushed immediately
# Important: queue the expected color before the product can reach the pusher sensor.
def wait_until_current_product_sorted(color, timeout=WAIT_SORT_DONE_TIMEOUT):
    print('Waiting until', color, 'reaches pusher/sort point')
    finished = sort_done_event.wait(timeout)

    if finished:
        print(color, 'moved to sorting point, next material can be emitted')
    else:
        print(color, 'sort wait timeout, next material will continue')

    sort_done_event.clear()
    tt.sleep(REPEAT_DELAY)
    return finished

def process_one_product(color):
    sort_done_event.clear()

    if not queue_product(color):
        tt.sleep(REPEAT_DELAY)
        return

    if color == 'blue':
        feed_mc0_and_run()
    elif color == 'green':
        feed_mc1_and_run()
    else:
        print('Unknown color skipped:', color)
        sort_done_event.set()
        return

    wait_until_current_product_sorted(color)

def production_loop():
    next_color = 'blue'
    print('Production loop started: queue first, process one material, then emit next')

    while process_run:
        if next_color == 'blue':
            process_one_product('blue')
            next_color = 'green'
        else:
            process_one_product('green')
            next_color = 'blue'

def sorting_loop():
    prev_active_sensors = []
    prev_exit = False
    print('Sorting loop started: blue is pushed immediately at pusher sensor')

    while process_run:
        sort_sensor, active_sensors = read_sort_sensor()
        exit_sensor = read_input(EXIT_SENSOR)

        if sort_sensor and not prev_active_sensors:
            expected_color = peek_product()
            print('Sort sensor detected:', active_sensors, 'expected=', expected_color)

            if expected_color == 'blue':
                if PUSHER_ACTIVATION_DELAY > 0:
                    tt.sleep(PUSHER_ACTIVATION_DELAY)
                write_coil(COIL_PUSHER_0, True)

            color = pop_product()
            print('Pusher sensor product:', color)

            if color == 'blue':
                finish_pusher_cycle()
            else:
                handle_sorting(color)

            sort_done_event.set()

        if exit_sensor and not prev_exit:
            print('Product reached exit')

        prev_active_sensors = active_sensors
        prev_exit = exit_sensor
        tt.sleep(SCAN_TIME)

def factory_process():
    global process_run, production_thread, sorting_thread, product_queue

    process_run = True
    product_queue = []
    sort_done_event.clear()

    reset_machines()
    conveyors_on()
    write_coil(COIL_PUSHER_0, False)

    print('Full process started: queue first, MC0=blue, MC1=green, pusher=blue only')

    production_thread = threading.Thread(target=production_loop, daemon=True)
    sorting_thread = threading.Thread(target=sorting_loop, daemon=True)

    production_thread.start()
    sorting_thread.start()

mc0 done busy_started= True activity= False progress= 0
mc1 green start= 12 produce= 13
mc1 busy=True progress= 0
mc1 done busy_started= True activity= True progress= 0
Repeated machining-center work stopped


In [72]:
# Cell 11: reset all outputs
stop_all()


All outputs stopped


In [73]:
# Cell 12: conveyor test
conveyors_on()
tt.sleep(3)
conveyors_off()


In [74]:
# Cell 13: emitter test
test_output(COIL_EMITTER_0)
test_output(COIL_EMITTER_1)


Coil 0 ON
Coil 0 OFF
Coil 9 ON
Coil 9 OFF


In [75]:
# Cell 14: pusher output test
# If this does not move the pusher, check Factory I/O Pusher 0 coil mapping.
test_pusher_only(seconds=1.0)


Testing Pusher 0 coil = 5
front= False back= False


In [76]:
# Cell 15: machine status monitor
print_machine_status(seconds=5)


MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0

In [77]:
# Cell 16: MC0 blue force test
force_mc0_test()


MC0 blue force test
mc0 busy=True progress= 0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=False, error=False, progress=0
MC0 blue o

In [78]:
# Cell 17: MC1 green force test
force_mc1_test()


MC1 green force test
mc1 busy=True progress= 0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=True, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=True, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=True, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=True, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=True, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=True, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=True, error=False, progress=0
MC0 blue opened=True, busy=True, error=False, progress=0 | MC1 green opened=True, busy=True, error=False, progress=0
MC0 blue opened=T

In [79]:
# Cell 18: sort-area input monitor
# Run this while blue passes in front of the pusher.
monitor_sort_area(seconds=20)


Watch which Retro Sensor turns True when blue reaches the pusher.
SORT_SENSOR = 11 candidates = [10, 11, 12]
Retro Sensor 0=True | Retro Sensor 1=True | Retro Sensor 2=True | Retro Sensor 3 / SORT_SENSOR default=True | Retro Sensor 4 / EXIT_SENSOR default=True | Pusher Front Limit=False | Pusher Back Limit=True queue= []
Retro Sensor 3 / SORT_SENSOR default=False queue= []
Retro Sensor 3 / SORT_SENSOR default=True queue= []
Retro Sensor 1=False | Retro Sensor 2=False queue= []
Retro Sensor 1=True queue= []


In [80]:
# Cell 19: start repeated machining-center work
machine_repeat_thread = threading.Thread(target=machine_repeat, kwargs={'mc': 'both'}, daemon=True)
machine_repeat_thread.start()


In [ ]:
# Cell 20: start full process
factory_process()


Repeated machining-center work started
Full process started: queue first, MC0=blue, MC1=green, pusher=blue only
Production loop started: queue first, process one material, then emit next
Queued: blue queue= ['blue']
Sorting loop started: blue is pushed immediately at pusher sensor


Sort sensor detected: [11, 12] expected= blue
Pusher sensor product: blue
Pusher cycle done
Product reached exit
Product reached exit
mc0 blue start= 3 produce= 4
mc0 blue start= 3 produce= 4
mc0 busy=True progress= 0
mc0 busy=True progress= 0
mc0 done busy_started= True activity= False progress= 0
mc0 done busy_started= True activity= False progress= 0
Waiting until blue reaches pusher/sort point
blue moved to sorting point, next material can be emitted
Queued: green queue= ['green']
mc1 green start= 12 produce= 13
mc1 green start= 12 produce= 13
mc1 busy=True progress= 0
mc1 busy=True progress= 0
Product reached exit
mc1 done busy_started= True activity= False progress= 0
mc1 done busy_started= True activity= False progress= 0
Waiting until green reaches pusher/sort point
mc0 blue start= 3 produce= 4
mc0 busy=True progress= 0
mc0 done busy_started= True activity= False progress= 0
Product reached exit
mc1 green start= 12 produce= 13
mc1 busy=True progress= 0
green sort wait timeout, 

In [ ]:
# Cell 21: stop everything
stop_all()


All outputs stopped


mc0 blue start= 3 produce= 4
blue sort wait timeout, next material will continue
mc0 busy did not start, progress= 0
mc0 done busy_started= False activity= False progress= 0
mc1 green start= 12 produce= 13
mc1 busy did not start, progress= 0
mc1 done busy_started= False activity= False progress= 0
Repeated machining-center work stopped
